# HTTP 接口与 FastAPI 运行

学习目标：写出一个小型 HTTP 接口，用客户端和浏览器调用，并说明应用与服务器的分工。

前置知识：Python 函数、装饰器、类型标注、字典、JSON 与终端操作。

适用版本：Python 3.12、FastAPI 0.141.1。工作目录为 content/Web与应用开发/FastAPI；第 6 节需要本地端口 8010，浏览器文档界面需要网络加载资源。

环境准备：[本课程环境说明](README.md)。

配套脚本：位于 scripts/01-http-api-and-fastapi。

1. [app.py](scripts/01-http-api-and-fastapi/app.py)：第 6 节由 Uvicorn 导入的最小应用。

前五节直接运行 Notebook。到第 6 节时，先按终端步骤启动服务，再运行该节代码；完成浏览器操作后关闭服务。

## 1 写一条路由

假设我们希望调用方发送姓名，程序返回一句问候。FastAPI() 创建应用；app.get("/hello") 将 GET 方法和 /hello 路径关联到函数。name 未提供时采用默认值。

返回类型 dict[str, str] 表示键和值都为字符串的字典，FastAPI 也会用它描述响应。下面只有一条路由；直接调用 hello() 是普通函数调用，还没有发送 HTTP 请求。

In [1]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/hello")
def hello(name: str = "读者") -> dict[str, str]:
    return {"message": f"你好，{name}"}


print(hello("小林"))  # 预期：{'message': '你好，小林'}。
# 返回 Python 字典；下一节观察它经过接口后形成的响应。

{'message': '你好，小林'}


## 2 发出一次请求

客户端负责发送请求、读取响应；FastAPI 应用负责找到路由并处理请求。TestClient 可以在当前 Python 进程内调用应用，便于立即观察结果，此时不需要监听网络端口。

get() 发出 GET 请求，status_code 是响应状态码，json() 将 JSON 响应解析为 Python 对象。with 会在代码块结束时关闭测试客户端。

In [2]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    response = client.get("/hello")
    print(response.status_code, response.json())  # 预期：200 {'message': '你好，读者'}。
    assert response.status_code == 200
    assert response.json() == {"message": "你好，读者"}
# 200 表示本次请求成功；观察返回内容与业务函数的关系。

200 {'message': '你好，读者'}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 3 看懂 URL 和方法

URL 指向请求目标。http://127.0.0.1:8010/hello?name=Lin 中，http 是协议，127.0.0.1:8010 是本地服务地址，/hello 是路径，name=Lin 是查询字符串。

HTTP 方法表达操作含义。这里用 GET 读取问候消息；params 让客户端构造查询字符串，查询部分不写入路由路径。下面继续使用本篇的 app。

In [3]:
with TestClient(app) as client:
    response = client.get("/hello", params={"name": "Lin"})
    print(response.request.method, response.request.url)  # 预期：GET http://testserver/hello?name=Lin。
    print(response.json())  # 预期：{'message': '你好，Lin'}。
    assert response.json() == {"message": "你好，Lin"}
# TestClient 使用测试地址；第 6 节改为真正的本地网络地址。

GET http://testserver/hello?name=Lin
{'message': '你好，Lin'}


## 4 请求头与请求体

请求头携带内容类型等附加信息，请求体承载提交的数据。Content-Type 描述内容格式，例如 application/json 表示 JSON。这里用 POST 提交数据并让接口回显，不保存数据。

为应用增加 /echo 路由。payload 的类型表示键和值都为字符串的字典，FastAPI 从请求体读取它；客户端的 json= 完成 JSON 编码并设置请求 Content-Type。

In [4]:
@app.post("/echo")
def echo(payload: dict[str, str]) -> dict[str, str]:
    return payload


with TestClient(app) as client:
    response = client.post("/echo", json={"topic": "HTTP"})
    print(response.request.headers["content-type"])  # 预期：application/json。
    print(response.headers["content-type"], response.json())  # 预期：application/json {'topic': 'HTTP'}。
    assert response.json() == {"topic": "HTTP"}
# 对照请求与响应的媒体类型，两者的内容均为 JSON。

application/json
application/json {'topic': 'HTTP'}


## 5 根据状态码判断结果

状态码说明请求的处理情况。2xx、4xx、5xx 分别指以 2、4、5 开头的三位状态码：2xx 表示成功，4xx 表示客户端请求有问题或无法满足，5xx 表示服务端处理发生错误。判断调用是否成功，要先看状态码。

404 表示没有找到目标资源；405 表示该目标不支持请求的方法。本应用只为 /hello 声明 GET，下面故意发出两种无效请求。

In [5]:
with TestClient(app) as client:
    missing = client.get("/missing")
    wrong_method = client.post("/hello")
    print(missing.status_code, missing.json())  # 预期：404 {'detail': 'Not Found'}。
    print(wrong_method.status_code, wrong_method.json())  # 预期：405 {'detail': 'Method Not Allowed'}。
    assert missing.status_code == 404
    assert wrong_method.status_code == 405
# 已核对预期失败；这两次响应并不是服务器启动失败。

404 {'detail': 'Not Found'}
405 {'detail': 'Method Not Allowed'}


## 6 用 Uvicorn 提供真实服务

让浏览器调用接口，需要服务器监听网络端口。FastAPI 应用负责接口逻辑，Uvicorn 负责监听端口并接收网络请求；两者通过 ASGI（异步服务器网关接口）配合。

配套 app.py 只保存前面两条路由。app:app 的两部分分别是模块名和应用对象名，--app-dir 指定模块搜索目录。

Step 1：在已激活课程环境、位于项目根目录的独立终端进入课程目录。

```powershell
cd content/Web与应用开发/FastAPI
```

Step 2：启动服务，等待出现 Application startup complete，并保持该终端运行。

```powershell
python -m uvicorn app:app --app-dir scripts/01-http-api-and-fastapi --host 127.0.0.1 --port 8010
```

HTTPX 是 HTTP 客户端库。下面的调用经过本地网络端口；timeout 限制网络等待，trust_env=False 让本地请求不使用环境变量中的代理设置。

In [6]:
import httpx

with httpx.Client(base_url="http://127.0.0.1:8010", timeout=5, trust_env=False) as client:
    response = client.get("/hello", params={"name": "Lin"})
    response.raise_for_status()
    print(response.status_code, response.json())  # 预期：200 {'message': '你好，Lin'}。
    assert response.json() == {"message": "你好，Lin"}
# 关闭 HTTPX 客户端后，终端里的 Uvicorn 服务仍在运行。

200

 {'message': '你好，Lin'}


## 7 查看接口文档并关闭服务

OpenAPI 是描述 HTTP 接口的规范。FastAPI 根据路由和参数声明生成 /openapi.json；/docs 使用 Swagger UI 展示接口并提供调用入口。

浏览器打开 http://127.0.0.1:8010/docs，展开 GET /hello，点击 Try it out，填写 name 为 Lin，再点击 Execute。核对状态码 200 和“你好，Lin”，这与第 6 节调用的是同一接口。

![Swagger UI 调用问候接口：请求 URL 含 name=Lin，状态码 200，响应体返回“你好，Lin”](image/01-swagger-response.png)

这张实际请求截图中，Request URL 显示发送到哪个地址及传入的姓名；Server response 中的 200 表示成功，Response body 展示实际返回的 JSON。下方 Responses 是接口声明，用来描述可能的响应。

默认文档界面从 CDN 加载资源，需要网络。下面直接读取本地 OpenAPI，观察它列出的两条路由。

In [7]:
with httpx.Client(base_url="http://127.0.0.1:8010", timeout=5, trust_env=False) as client:
    response = client.get("/openapi.json")
    response.raise_for_status()
    paths = response.json()["paths"]
    print({path: list(methods) for path, methods in paths.items()})  # 预期：{'/hello': ['get'], '/echo': ['post']}。
    assert "get" in paths["/hello"] and "post" in paths["/echo"]
    hello_response = paths["/hello"]["get"]["responses"]["200"]
    assert hello_response["content"]["application/json"]["schema"]["type"] == "object"
# 问候接口的返回类型让文档将响应描述为 JSON 对象。
# 浏览器操作完成后，在服务终端按 Ctrl+C，等待进程退出。
# 关闭标签页或客户端连接，不会自动停止 Uvicorn。

{'/hello': ['get'], '/echo': ['post']}


## 本章小结

（1）路由由方法与路径定位；函数处理输入并返回结果。

（2）读请求时区分 URL、请求头与请求体；读响应时同时核对状态码和内容。

（3）TestClient 便于进程内观察，Uvicorn 提供真实网络服务；OpenAPI 和 Swagger UI 帮助描述与调用接口。

## 练习

（1）新增 GET /health，返回 {"status": "ok"}。用 TestClient 核对状态码与内容，再将路由加入配套服务并重启，用浏览器调用。

（2）分别省略 name 和传入一个新姓名。预测两个响应后再调用，确认默认值只在未提供参数时使用。

（3）通过浏览器文档向 /echo 提交两个字符串字段，核对回显内容。完成后关闭服务，再刷新 /hello，确认服务已停止。

## 参考与引用来源

- FastAPI 官方文档：[First Steps](https://fastapi.tiangolo.com/tutorial/first-steps/)，定位路径操作、返回内容、Interactive API docs 与 OpenAPI；[Testing](https://fastapi.tiangolo.com/tutorial/testing/)，定位 TestClient；[Query Parameters](https://fastapi.tiangolo.com/tutorial/query-params/)，定位默认参数；[Bodies of arbitrary dicts](https://fastapi.tiangolo.com/tutorial/body-nested-models/#bodies-of-arbitrary-dicts)，定位字典请求体；[Response Model - Return Type](https://fastapi.tiangolo.com/tutorial/response-model/)，定位开篇返回类型声明与 OpenAPI 响应模式；[Custom Docs UI Static Assets](https://fastapi.tiangolo.com/how-to/custom-docs-ui-assets/)，定位默认 CDN 资源。
- ASGI 官方规范：[Introduction](https://asgi.readthedocs.io/en/latest/introduction.html)，定位 Rationale 与 Specification Details，说明服务器与应用的接口。
- RFC Editor：[RFC 9110](https://www.rfc-editor.org/rfc/rfc9110.html)，定位第 4、6、8.3、9 与 15 节，支持 URL、消息组成、Content-Type、方法与状态码。
- HTTPX 官方文档：[QuickStart](https://www.python-httpx.org/quickstart/)，定位查询参数、JSON、状态码与响应头；[Environment Variables](https://www.python-httpx.org/environment_variables/)，定位 trust_env。
- Uvicorn 官方文档：[Settings](https://uvicorn.dev/settings/)，定位 Application 与 Socket Binding；[Deployment](https://uvicorn.dev/deployment/)，定位运行与进程管理。命令适配本课程目录与 Conda 环境。